In [ ]:
!pip install langchain

### First LangChain Code
- Install langchain-ollama

In [ ]:
!pip install langchain-ollama

### First chat with Ollama using Langchain

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

template = PromptTemplate.from_template("What is the capital of {countryName}?")
prompt = template.invoke({"countryName": "France"})


chatTemplate = ChatPromptTemplate.from_messages()

print(prompt)

llm_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '1083e4f0a13244188a85d64f157f5264.VJIsvloP8AwhWbdyQt_asVPy'}
        }
    )

res = llm_model.invoke(prompt)
print(res)



text='What is the capital of France?'
content='The capital of France is Paris.' additional_kwargs={} response_metadata={'model': 'qwen3-coder-next:cloud', 'created_at': '2026-03-31T14:38:04.67813882Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2231798628, 'load_duration': None, 'prompt_eval_count': 15, 'prompt_eval_duration': None, 'eval_count': 8, 'eval_duration': None, 'logprobs': None, 'model_name': 'qwen3-coder-next:cloud', 'model_provider': 'ollama'} id='lc_run--019d4454-7c61-7813-a090-cb022bf0f38e-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 15, 'output_tokens': 8, 'total_tokens': 23}


### Different Types of Messages

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# prompt = "What is the capital of France?"  
# print(type(prompt))  # string

# humanMsg = HumanMessage(content="what is the capital of France?")
# print(type(humanMsg))

# print(humanMsg)

llm_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '1083e4f0a13244188a85d64f157f5264.VJIsvloP8AwhWbdyQt_asVPy'}
        }
    )

# res = llm_model.invoke(humanMsg.content)
# print(res)
# print(type(res))




### ChatPromptTemplate

chatTemplate = ChatPromptTemplate.from_messages([
    ("human", "What is the capital of {countryName}?"),
    ("ai", "The capital of {countryName} is {cityName}"),
    ("human","What is the most popular city of {cityName}")

    # HumanMessage(content="What is the capital of {countryName}?"),
    # AIMessage(content= "The capital of {countryName} is {cityName}"),
    # HumanMessage(content= "What is the most popular city of {cityName}")
])

#res = llm_model.invoke(chatTemplate.invoke({"countryName": "India", "cityName": "Delhi"}))

chain = chatTemplate | llm_model

res = chain.invoke({"countryName": "India", "cityName": "Delhi"})

print(res)


content='Delhi itself is a city—but it\'s actually divided into two main parts:\n\n- **New Delhi**: The planned, northern part of Delhi, which serves as the **national capital territory** and houses government institutions (like the Parliament, Rashtrapati Bhavan, and Supreme Court). It\'s often what people mean when they refer to "Delhi" in an administrative or political context.\n\n- **Old Delhi**: The historic, southern part of Delhi, known for its Mughal-era landmarks like the Red Fort, India Gate, Jama Masjid, and bustling markets like Chandni Chowk.\n\nSo, there isn’t a “most popular city *of* Delhi”—rather, **New Delhi and Old Delhi** are the two iconic regions *within* the larger city of **Delhi** (officially the **National Capital Territory of Delhi**, or NCT).\n\nDelhi is also India’s **second-most populous city** (after Mumbai), and it’s a major cultural, political, and economic hub.\n\nLet me know if you\'d like highlights of Old vs. New Delhi! 😊' additional_kwargs={} respo

### Understanding Tooling in LangChain

In [25]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompt_values import PromptValue
from pydantic import BaseModel

class AIResponse(BaseModel):
    countryName: str
    countryWeather: str
    countryCapital: str    


@tool
def get_weather_city(city: str) -> str:
    """Get the current weather for a city
    
    Args:
        city: The name of the city for which to get weather information
    
    Returns:
        A string describing the weather in the specified city
    """
    return f"The weather in {city} is sunny."

@tool
def calculate(expression: str) -> str:
    """Calculate the result of a mathematical expression
    
    Args:
        expression: A string representing a mathematical expression
    
    Returns:
        A string representing the result of the calculation
    """
    return str(eval(expression))



def podTestAgent(prompt: ChatPromptTemplate, format: any, countryName: str ):  

    promptValue = prompt.invoke({"format": format, "countryName": countryName})  
    messages = list(promptValue.to_messages())

    tools = [get_weather_city, calculate]
    tools_by_name = {}   # str: StructureTool  {"get_weather_city":  StructuredTool}

    for tool in tools:
        tools_by_name[f"{tool.name}"] = tool   

    llm_model = ChatOllama(
        base_url="https://ollama.com",
        model="qwen3-coder-next:cloud",
        client_kwargs=  {  
            "headers": {'Authorization': 'Bearer ' + '1083e4f0a13244188a85d64f157f5264.VJIsvloP8AwhWbdyQt_asVPy'}
            }
        )

    llm_model_with_tools = llm_model.bind_tools(tools)


    while True:
        res = llm_model_with_tools.invoke(messages)
        print(res)
        #print(str(prompt.invoke()))

        if not res.tool_calls:
            return res.content

        # extract tool_call info
        tools_call = res.tool_calls  # list of tools
        for tool in tools_call:
            tool_name = tool["name"]
            tool_args =tool["args"]
            tool_id = tool["id"]
            tool_result = tools_by_name[tool_name].invoke(tool_args)
            #print(tool_result)

            toolmsg = ToolMessage(content=str(tool_result), tool_call_id=tool_id)
            messages.append(toolmsg)          
   


prompt = ChatPromptTemplate.from_messages([   
     ("system", "Provide response in JSON format {format}"),
    ("human", "What is the weather in {countryName} right now?   what is it's capital? ")    
    ])

res = podTestAgent( prompt,  AIResponse.model_json_schema(), "India")
print(res)


#print(type(res))


content='' additional_kwargs={} response_metadata={'model': 'qwen3-coder-next:cloud', 'created_at': '2026-04-02T14:29:29.158935244Z', 'done': True, 'done_reason': 'stop', 'total_duration': 567166761, 'load_duration': None, 'prompt_eval_count': 480, 'prompt_eval_duration': None, 'eval_count': 24, 'eval_duration': None, 'logprobs': None, 'model_name': 'qwen3-coder-next:cloud', 'model_provider': 'ollama'} id='lc_run--019d4e99-6142-7d11-9308-ca7ff56ee7a7-0' tool_calls=[{'name': 'get_weather_city', 'args': {'city': 'New Delhi'}, 'id': '4a397311-6c63-4891-a750-414934e2671a', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 480, 'output_tokens': 24, 'total_tokens': 504}
content='' additional_kwargs={} response_metadata={'model': 'qwen3-coder-next:cloud', 'created_at': '2026-04-02T14:29:30.242972546Z', 'done': True, 'done_reason': 'stop', 'total_duration': 602832138, 'load_duration': None, 'prompt_eval_count': 496, 'prompt_eval_duration': None, 'eval_count': 24, 'eva

KeyboardInterrupt: 

In [28]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompt_values import PromptValue
from pydantic import BaseModel
from langchain_core.runnables import RunnableSequence

class AIResponse(BaseModel):
    countryName: str
    countryWeather: str
    countryCapital: str 

prompt = ChatPromptTemplate.from_messages([   
     ("system", "Provide response in JSON format {format}"),
    ("human", "What is the weather in {countryName} right now?   what is it's capital? ")    
    ])

#input  {"format": AIResponse.model_json_schema(), "countryName": "India"}
#Output of ChatPromptTemplate: promptValue
#promptValue = prompt.invoke({"format": AIResponse.model_json_schema(), "countryName": "India"}) 

llm_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '1083e4f0a13244188a85d64f157f5264.VJIsvloP8AwhWbdyQt_asVPy'}
        }
    )  #.with_structured_output(AIResponse)

#chaining of Runnables:
#chain =  prompt | llm_model
chain = RunnableSequence(prompt, llm_model)
res = chain.invoke({"format": AIResponse.model_json_schema(), "countryName": "India"})

#RunnableSequnce
#RunnableSequence(prompt, llm_model)

#res = llm_model.invoke(promptValue)



print(res)





content='{\n  "countryName": "India",\n  "countryWeather": "I cannot provide real-time weather data. Please check a reliable weather service like Weather.com, AccuWeather, or a weather app for current conditions in India.",\n  "countryCapital": "New Delhi"\n}' additional_kwargs={} response_metadata={'model': 'qwen3-coder-next:cloud', 'created_at': '2026-04-02T14:41:36.428492641Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1034604247, 'load_duration': None, 'prompt_eval_count': 118, 'prompt_eval_duration': None, 'eval_count': 58, 'eval_duration': None, 'logprobs': None, 'model_name': 'qwen3-coder-next:cloud', 'model_provider': 'ollama'} id='lc_run--019d4ea4-786f-7392-8dcc-77fc735070a9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 118, 'output_tokens': 58, 'total_tokens': 176}


### Understand Parallel Execution of runnables

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompt_values import PromptValue
from pydantic import BaseModel
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnableLambda

class AIResponse(BaseModel):
    countryName: str
    countryWeather: str
    countryCapital: str 

prompt = ChatPromptTemplate.from_messages([   
     ("system", "Provide response in JSON format {format}"),
    ("human", "What is the weather in {countryName} right now?   what is it's capital? ")    
    ])

llm_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '1083e4f0a13244188a85d64f157f5264.VJIsvloP8AwhWbdyQt_asVPy'}
        }
    )  #.with_structured_output(AIResponse)

prompt2 = PromptTemplate.from_template("who is the Prime minister of Canada?")

#chaining of Runnables:
#chain =  prompt | llm_model
chain1 = RunnableSequence(prompt, llm_model)
chain2= RunnableSequence(prompt2, llm_model)

finalchain = RunnableParallel(
    country_info=chain1,   # result of chain1 stored under key "country_info"
    pm_info=chain2  )

res = finalchain.invoke({"format": AIResponse.model_json_schema(), "countryName": "India"})
#res = chain.invoke({"format": AIResponse.model_json_schema(), "countryName": "India"})

#RunnableSequnce
#RunnableSequence(prompt, llm_model)

#res = llm_model.invoke(promptValue)

def add(a: int, b: int) -> int:
    return a+b



#res =  | llm_model


print(res)





{'country_info': AIMessage(content='{"countryName": "India", "countryWeather": "Partly cloudy", "countryCapital": "New Delhi"}', additional_kwargs={}, response_metadata={'model': 'qwen3-coder-next:cloud', 'created_at': '2026-04-02T14:53:01.343123729Z', 'done': True, 'done_reason': 'stop', 'total_duration': 702128727, 'load_duration': None, 'prompt_eval_count': 118, 'prompt_eval_duration': None, 'eval_count': 25, 'eval_duration': None, 'logprobs': None, 'model_name': 'qwen3-coder-next:cloud', 'model_provider': 'ollama'}, id='lc_run--019d4eae-ecfe-7d50-9603-643e61fcdf49-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 118, 'output_tokens': 25, 'total_tokens': 143}), 'pm_info': AIMessage(content='As of now, the Prime Minister of Canada is **Justin Trudeau**. He has held the position since November 4, 2015, and was most recently re-elected in the 2021 federal election. He leads the Liberal Party of Canada.\n\nNote: Political leadership can change, so for the most u

In [33]:
from langchain_core.runnables import RunnableLambda

def add(a: int, b: int) -> int:
    return a+b

r1 = RunnableLambda(lambda x: add(**x))
res = r1.invoke({"a": 4, "b": 6})
print(res)

#c = r1 | llm_model



10


### Easiest way to create an Agents